# Cell 1 — Import Libraries

In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

# Cell 2 — Load Dataset (Chunk Loading)

The dataset is very large, so we load it in **chunks**.

In [2]:
chunk_size = 200000
chunks = []

for chunk in pd.read_csv("../data/raw/DNN-EdgeIIoT-dataset.csv", chunksize=chunk_size):
    chunks.append(chunk)

df = pd.concat(chunks)

print("Dataset Shape:", df.shape)

C:\Users\nimma\AppData\Local\Temp\ipykernel_22196\2651902167.py:4: DtypeWarning: Columns (0: arp.dst.proto_ipv4, 1: arp.src.proto_ipv4) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv("../data/raw/DNN-EdgeIIoT-dataset.csv", chunksize=chunk_size):
C:\Users\nimma\AppData\Local\Temp\ipykernel_22196\2651902167.py:4: DtypeWarning: Columns (0: arp.dst.proto_ipv4, 1: arp.src.proto_ipv4) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv("../data/raw/DNN-EdgeIIoT-dataset.csv", chunksize=chunk_size):
C:\Users\nimma\AppData\Local\Temp\ipykernel_22196\2651902167.py:4: DtypeWarning: Columns (0: arp.dst.proto_ipv4, 1: arp.src.proto_ipv4) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv("../data/raw/DNN-EdgeIIoT-dataset.csv", chunksize=chunk_size):
C:\Users\nimma\AppData\Local\Temp\ipykernel_22196\2651902167.py:4: DtypeWarning: Columns (0: arp.dst

Dataset Shape: (2219201, 63)


# Cell 3 — Check Attack Distribution

In [3]:
print(df["Attack_type"].value_counts())

print("\nPercentages:")
print(df["Attack_type"].value_counts(normalize=True))

Attack_type
Normal                   1615643
DDoS_UDP                  121568
DDoS_ICMP                 116436
SQL_injection              51203
Password                   50153
Vulnerability_scanner      50110
DDoS_TCP                   50062
DDoS_HTTP                  49911
Uploading                  37634
Backdoor                   24862
Port_Scanning              22564
XSS                        15915
Ransomware                 10925
MITM                        1214
Fingerprinting              1001
Name: count, dtype: int64

Percentages:
Attack_type
Normal                   0.728029
DDoS_UDP                 0.054780
DDoS_ICMP                0.052468
SQL_injection            0.023073
Password                 0.022600
Vulnerability_scanner    0.022580
DDoS_TCP                 0.022559
DDoS_HTTP                0.022491
Uploading                0.016958
Backdoor                 0.011203
Port_Scanning            0.010168
XSS                      0.007172
Ransomware               0.004923

# Cell 4 — Drop Non-Useful Columns (Paper-style preprocessing)

These columns contain packet identifiers or payload data.

In [4]:
drop_columns = [
"frame.time",
"ip.src_host",
"ip.dst_host",
"arp.src.proto_ipv4",
"arp.dst.proto_ipv4",
"http.file_data",
"http.request.full_uri",
"http.request.uri.query",
"icmp.transmit_timestamp",
"tcp.options",
"tcp.payload",
"tcp.srcport",
"tcp.dstport",
"udp.port",
"mqtt.msg"
]

df.drop(columns=drop_columns, inplace=True, errors="ignore")

# Cell 5 — Remove Missing Values and Duplicates

In [5]:
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

df = shuffle(df, random_state=42)

print("Shape after cleaning:", df.shape)

Shape after cleaning: (1924552, 48)


# Cell 6 — Save Cleaned Dataset

This file will also be used for **PPFLE encoding later**.

In [6]:
df.to_csv("../data/processed/dl/dl_cleaned.csv", index=False)

# Cell 7 — Separate Features and Labels

Important: remove **Attack_label** to avoid leakage.

In [7]:
X = df.drop(columns=["Attack_type", "Attack_label"])
y = df["Attack_type"]

# Cell 8 — Encode Categorical Features

In [8]:
categorical_columns = [
"http.request.method",
"http.referer",
"http.request.version",
"dns.qry.name.len",
"mqtt.conack.flags",
"mqtt.protoname",
"mqtt.topic"
]

X = pd.get_dummies(X, columns=categorical_columns)

print("Total features after encoding:", X.shape[1])

Total features after encoding: 103


# Cell 9 — Save Encoded Dataset

In [9]:
encoded_df = pd.concat([X, y], axis=1)

encoded_df.to_csv("../data/processed/dl/dl_encoded.csv", index=False)

# Cell 10 — Train/Test Split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Cell 11 — Save Train/Test Split

In [11]:
data_split = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test
}

with open("../data/processed/dl/dl_train_test_split.pkl", "wb") as f:
    pickle.dump(data_split, f)